# Estación Meteorológica IoT — CC3067 Redes (topic 23415)
**Entorno de ejecución → Ejecutar todo.** La última celda corre el productor (hilo en segundo plano) y el consumidor con la gráfica en vivo durante unos minutos. Al final queda la gráfica y se guarda `grafica.png`.

In [ ]:
!pip install -q confluent-kafka

In [ ]:
# ===== Configuración, sensores y encode/decode =====
"""Configuración compartida, simulación de sensores y codificación de 3 bytes."""
import json
import random

# ---------------- Kafka ----------------
BOOTSTRAP = "161.35.11.141:9092"
TOPIC = "23415"          # carné como topic único
KEY = "sensor1"

KAFKA_CONF = {
    "bootstrap.servers": BOOTSTRAP,
    "security.protocol": "SASL_PLAINTEXT",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": "estudiante",
    "sasl.password": "TryluRandomPass-2!@!",
}

# ---------------- Sensores ----------------
DIRECCIONES = ["N", "NO", "O", "SO", "S", "SE", "E", "NE"]  # 8 opciones -> 3 bits
MEDIA = 55
DESV = 15  # varianza = 225


def _clamp(x, lo, hi):
    return max(lo, min(hi, x))


def generar_data():
    """Muestrea de una normal(55, 15) y recorta a los rangos de cada sensor."""
    temperatura = round(_clamp(random.gauss(MEDIA, DESV), 0, 110), 2)
    humedad = int(round(_clamp(random.gauss(MEDIA, DESV), 0, 100)))
    direccion = random.choice(DIRECCIONES)
    return {"temperatura": temperatura, "humedad": humedad, "direccion_viento": direccion}


# ---------------- JSON ----------------
def to_json(data):
    return json.dumps(data).encode("utf-8")


def from_json(raw):
    return json.loads(raw.decode("utf-8"))


# ---------------- Payload restringido: 3 bytes ----------------
# | temperatura*100 (14 bits) | humedad (7 bits) | viento (3 bits) |  = 24 bits
# 110.00 * 100 = 11000 < 2^14 = 16384, así que cabe como entero.

def encode(data):
    temp = int(round(data["temperatura"] * 100))
    hum = int(data["humedad"])
    viento = DIRECCIONES.index(data["direccion_viento"])
    assert 0 <= temp < 2**14 and 0 <= hum < 2**7 and 0 <= viento < 2**3
    packed = (temp << 10) | (hum << 3) | viento
    return packed.to_bytes(3, "big")


def decode(raw):
    packed = int.from_bytes(raw, "big")
    return {
        "temperatura": ((packed >> 10) & 0x3FFF) / 100,
        "humedad": (packed >> 3) & 0x7F,
        "direccion_viento": DIRECCIONES[packed & 0x7],
    }


def parse_mensaje(raw):
    """Detecta el formato: 3 bytes -> decode compacto, si no -> JSON."""
    return decode(raw) if len(raw) == 3 else from_json(raw)


In [ ]:
# ===== Productor (dispositivo IoT) en un hilo =====
import threading, time, random
from confluent_kafka import Producer, Consumer, KafkaError

COMPACT = False        # True = payload de 3 bytes (sección 3.4)
INTERVALO = (3, 6)     # segundos entre envíos para la demo (el lab pide 15-30)
DURACION = 180         # segundos que corre la demo

stop = threading.Event()

def productor():
    p = Producer(KAFKA_CONF)
    while not stop.is_set():
        d = generar_data()
        payload = encode(d) if COMPACT else to_json(d)
        p.produce(TOPIC, key=KEY, value=payload)
        p.flush(10)
        print(f"[PRODUCER] {d} -> {payload!r} ({len(payload)} bytes)")
        stop.wait(random.uniform(*INTERVALO))
    p.flush(10)

threading.Thread(target=productor, daemon=True).start()


In [ ]:
# ===== Consumidor con gráfica en vivo =====
import matplotlib.pyplot as plt
from IPython.display import clear_output

conf = dict(KAFKA_CONF)
conf.update({"group.id": f"{TOPIC}-colab-{int(time.time())}",
             "auto.offset.reset": "latest", "enable.auto.commit": False})
consumer = Consumer(conf)
consumer.subscribe([TOPIC])

temps, hums, winds, log = [], [], [], []

def graficar():
    clear_output(wait=True)
    fig, (a, b, c) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
    fig.suptitle(f"Estación meteorológica IoT — topic {TOPIC} — {'3 bytes' if COMPACT else 'JSON'}")
    x = list(range(1, len(temps) + 1))
    a.plot(x, temps, marker="o", color="tab:red"); a.set_ylabel("Temperatura (°C)"); a.set_ylim(0, 110)
    b.plot(x, hums, marker="o", color="tab:blue"); b.set_ylabel("Humedad (%)"); b.set_ylim(0, 100)
    c.scatter(x, [DIRECCIONES.index(w) for w in winds], color="tab:green")
    c.set_yticks(range(8)); c.set_yticklabels(DIRECCIONES); c.set_ylabel("Dir. viento"); c.set_xlabel("Lectura #")
    for ax in (a, b, c): ax.grid(alpha=0.3)
    if temps:
        a.set_title(f"Última: {temps[-1]:.2f} °C | {hums[-1]} % | {winds[-1]}   (n={len(temps)})", fontsize=10)
    fig.tight_layout()
    fig.savefig("grafica.png", dpi=150)
    plt.show()
    print("\n".join(log[-6:]))

fin = time.time() + DURACION
try:
    while time.time() < fin:
        msg = consumer.poll(1.0)
        if msg is None: continue
        if msg.error():
            if msg.error().code() != KafkaError._PARTITION_EOF: print("[ERROR]", msg.error())
            continue
        d = parse_mensaje(msg.value())
        temps.append(d["temperatura"]); hums.append(d["humedad"]); winds.append(d["direccion_viento"])
        log.append(f"[CONSUMER] offset {msg.offset()}: {msg.value()!r} -> {d}")
        graficar()
finally:
    stop.set()
    consumer.close()
    print("Listo. Gráfica guardada en grafica.png")
